<a href="https://colab.research.google.com/github/EberHernandezBenitez/EDP/blob/main/L%C3%ADnea_espera_paralelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema de línea de espera con dos servidores en paralelo.

Consideremos un modelo en el que los clientes llegan a un sistema con dos servidores.

Al llegar un lciente, se forma en una fila si ambos servidores están ocupados, entra al servidor 1 ssi esta desocupado o al servidor 2 en caso contrario.

Cuando el cliente concluye el servicio con un servidor sale del sistema y el cliente que ha estado formado más tiempo entra a servicio.

Queremos un modelo en el cual se lleve un registro de las cantidades de tiempo que pasa cada cliente dentro del sistema y el número de servicios realizados por cada servidor.

Es importante señalar que los clientes no saldrán como fueron llegando.


En este modelo, cuando un cliente llega:

- Si ambos servidores están libres, va al Servidor 1 (por convención).

- Si solo uno está libre, va a ese servidor.

- Si ambos están ocupados, se une a una única fila de espera común.

Parámetros:
- T: Tiempo máximo de simulación (límite para aceptar llegadas).
- tasa_arribo: Parámetro lambda para las llegadas.
    
- tasa_servicio1: Parámetro mu para el servidor 1.
    
- tasa_servicio2: Parámetro mu para el servidor 2.

In [4]:
import random

def simular_servidores_en_paralelo(T, tasa_arribo, tasa_servicio1, tasa_servicio2):
    """
    Simulación de un sistema de línea de espera con dos servidores en paralelo.
    """
    # INICIALIZACIÓN (Basada en las variables de la sección 6.4)
    t = 0         # Variable de tiempo (reloj)
    N_A = 0       # Número de llegadas al sistema
    N_D = 0       # Número de salidas del sistema
    n = 0         # Clientes totales en el sistema (en cola + en servicio)

    # Estado de ocupación de los servidores (0 = libre, 1 = ocupado)
    # Si n=1, va al servidor 1.
    # Si n >= 2, ambos servidores están ocupados y hay (n-2) en cola.
    servidor1_ocupado = 0
    servidor2_ocupado = 0

    # Generar el primer arribo
    t_A = random.expovariate(tasa_arribo)

    # Tiempos de finalización de servicio (infinito si están vacíos)
    t1 = float('inf')
    t2 = float('inf')

    # Diccionarios para registrar los tiempos individuales de los clientes
    A = {}       # Hora de llegada del cliente i al sistema
    D = {}       # Hora de salida del cliente i del sistema
    S_tipo = {}  # Guarda qué servidor atendió al cliente (1 o 2) para estadísticas

    # Cola para mantener el orden de llegada
    cola = []

    # Rastrear qué cliente está siendo atendido en cada servidor
    cliente_en_s1 = None
    cliente_en_s2 = None

    # BUCLE PRINCIPAL DE SIMULACIÓN
    while t_A < float('inf') or t1 < float('inf') or t2 < float('inf'):

        proximo_evento = min(t_A, t1, t2)

        #CASO 1: Llegada de un cliente al sistema
        if proximo_evento == t_A:
            t = t_A
            N_A += 1
            n += 1

            # Generar la siguiente llegada
            T_r = random.expovariate(tasa_arribo)
            if t + T_r <= T:
                t_A = t + T_r
            else:
                t_A = float('inf')  # No se permiten más llegadas después de T

            A[N_A] = t  # Registrar tiempo de llegada del cliente N_A

            # Asignación de servidor
            if servidor1_ocupado == 0:
                # El servidor 1 está libre, entra de inmediato
                servidor1_ocupado = 1
                cliente_en_s1 = N_A
                S_tipo[N_A] = 1
                Y1 = random.expovariate(tasa_servicio1)
                t1 = t + Y1
            elif servidor2_ocupado == 0:
                # El servidor 1 está ocupado pero el 2 está libre
                servidor2_ocupado = 1
                cliente_en_s2 = N_A
                S_tipo[N_A] = 2
                Y2 = random.expovariate(tasa_servicio2)
                t2 = t + Y2
            else:
                # Ambos servidores ocupados, va a la fila
                cola.append(N_A)

        #CASO 2: Fin de servicio en el Servidor 1
        elif proximo_evento == t1:
            t = t1
            N_D += 1
            n -= 1

            # Registrar salida del cliente que estaba en el Servidor 1
            D[cliente_en_s1] = t

            if len(cola) == 0:
                # No hay nadie en cola, el servidor se libera
                servidor1_ocupado = 0
                cliente_en_s1 = None
                t1 = float('inf')
            else:
                # El cliente que más tiempo lleva esperando entra al Servidor 1
                siguiente_cliente = cola.pop(0)
                cliente_en_s1 = siguiente_cliente
                S_tipo[siguiente_cliente] = 1
                Y1 = random.expovariate(tasa_servicio1)
                t1 = t + Y1

        #CASO 3: Fin de servicio en el Servidor 2
        elif proximo_evento == t2:
            t = t2
            N_D += 1
            n -= 1

            # Registrar salida del cliente que estaba en el Servidor 2
            D[cliente_en_s2] = t

            if len(cola) == 0:
                # No hay nadie en cola, el servidor se libera
                servidor2_ocupado = 0
                cliente_en_s2 = None
                t2 = float('inf')
            else:
                # El cliente que más tiempo lleva esperando entra al Servidor 2
                siguiente_cliente = cola.pop(0)
                cliente_en_s2 = siguiente_cliente
                S_tipo[siguiente_cliente] = 2
                Y2 = random.expovariate(tasa_servicio2)
                t2 = t + Y2

    return A, D, S_tipo

def calcular_estadisticas_paralelo(A, D, S_tipo):
    """
    Vamos a calcular los tiempos promedio en el sistema ycuántos
    fueron atendidos por cada servidor.
    """
    total_atendidos = len(D)
    if total_atendidos == 0:
        print("Ningún cliente completó su servicio.")
        return

    suma_tiempo_sistema = 0.0
    suma_s1 = 0.0
    suma_s2 = 0.0
    cont_s1 = 0
    cont_s2 = 0

    for id_cliente, t_salida in D.items():
        t_llegada = A[id_cliente]
        t_sistema = t_salida - t_llegada
        suma_tiempo_sistema += t_sistema

        if S_tipo[id_cliente] == 1:
            suma_s1 += t_sistema
            cont_s1 += 1
        else:
            suma_s2 += t_sistema
            cont_s2 += 1

    promedio_global = suma_tiempo_sistema / total_atendidos
    promedio_s1 = (suma_s1 / cont_s1) if cont_s1 > 0 else 0
    promedio_s2 = (suma_s2 / cont_s2) if cont_s2 > 0 else 0

    return promedio_global, promedio_s1, promedio_s2, cont_s1, cont_s2

# EJEMPLO DE USO Y PRUEBA
if __name__ == "__main__":
    # Configuración de los parámetros
    TIEMPO_MAX = 100       # Tiempo máximo de arribos (T)
    LAMBDA_ARR = 3.5       # Tasa de arribos global (λ)
    MU_SERV1   = 2.0       # Capacidad de servicio del Servidor 1 (μ1)
    MU_SERV2   = 2.0       # Capacidad de servicio del Servidor 2 (μ2)

    # Ejecutar simulación
    llegadas, salidas, servidores = simular_servidores_en_paralelo(
        TIEMPO_MAX, LAMBDA_ARR, MU_SERV1, MU_SERV2
    )

    # Calcular promedios
    prom_global, prom_s1, prom_s2, c1, c2 = calcular_estadisticas_paralelo(llegadas, salidas, servidores)

    print(f"Total de clientes completamente atendidos: {len(salidas)}")
    print(f" -> Atendidos por el Servidor 1: {c1}")
    print(f" -> Atendidos por el Servidor 2: {c2}")

    print("="*55)
    print(f"Tiempo prom. total en el SISTEMA (Fila + Servicio): {prom_global:.4f}")
    print(f"Tiempo prom. en el sistema para usuarios del Servidor 1: {prom_s1:.4f}")
    print(f"Tiempo prom. en el sistema para usuarios del Servidor 2: {prom_s2:.4f}")
    print("="*55)

Total de clientes completamente atendidos: 322
 -> Atendidos por el Servidor 1: 171
 -> Atendidos por el Servidor 2: 151
Tiempo prom. total en el SISTEMA (Fila + Servicio): 1.0540
Tiempo prom. en el sistema para usuarios del Servidor 1: 0.9810
Tiempo prom. en el sistema para usuarios del Servidor 2: 1.1366
